In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv(r'C:\Users\juanb\Desktop\amz_uk.csv', low_memory=False)
df = df.dropna(subset=['stars', 'price', 'category'])
df = df[df['price'] > 0]

print(f"Filas tras limpieza: {df.shape[0]:,}")
print(f"Categorías únicas: {df['category'].nunique()}")

In [ ]:
freq_table = df['category'].value_counts().reset_index()
freq_table.columns = ['category', 'count']
freq_table['percentage'] = (freq_table['count'] / len(df) * 100).round(2)

print("TOP 10 CATEGORÍAS MÁS LISTADAS")
print("=" * 50)
print(freq_table.head(10).to_string(index=False))

In [ ]:
top15 = freq_table.head(15)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(top15['category'][::-1], top15['count'][::-1], color='steelblue')
ax.set_xlabel('Número de productos')
ax.set_title('Top 15 categorías más listadas en Amazon UK', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars[::-1], top15['count'][::-1]):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

top10 = freq_table.head(10).copy()
otros = pd.DataFrame([['Otros', len(df) - top10['count'].sum(), 0]],
                      columns=['category', 'count', 'percentage'])
top10_pie = pd.concat([top10, otros])

fig, ax = plt.subplots(figsize=(10, 7))
ax.pie(top10_pie['count'], labels=top10_pie['category'],
       autopct='%1.1f%%', startangle=140, pctdistance=0.85,
       textprops={'fontsize': 8})
ax.set_title('Proporción de productos: Top 10 categorías vs resto', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
mean_price = df['price'].mean()
median_price = df['price'].median()
mode_price = df['price'].mode()[0]

print("MEDIDAS DE CENTRALIDAD — PRECIO")
print("=" * 40)
print(f"Media:    £{mean_price:.2f}")
print(f"Mediana:  £{median_price:.2f}")
print(f"Moda:     £{mode_price:.2f}")

variance = df['price'].var()
std = df['price'].std()
price_range = df['price'].max() - df['price'].min()
q1 = df['price'].quantile(0.25)
q3 = df['price'].quantile(0.75)
iqr = q3 - q1

print("\nMEDIDAS DE DISPERSIÓN — PRECIO")
print("=" * 40)
print(f"Varianza:  £{variance:.2f}")
print(f"Desv. típica: £{std:.2f}")
print(f"Rango:     £{price_range:.2f}")
print(f"Q1:        £{q1:.2f}")
print(f"Q3:        £{q3:.2f}")
print(f"IQR:       £{iqr:.2f}")
print(f"Precio mín: £{df['price'].min():.2f}")
print(f"Precio máx: £{df['price'].max():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

p95 = df['price'].quantile(0.95)
df_filtered = df[df['price'] <= p95]

axes[0].hist(df_filtered['price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de precios (hasta percentil 95)', fontweight='bold')
axes[0].set_xlabel('Precio (£)')
axes[0].set_ylabel('Número de productos')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{int(x):,}'))
axes[0].axvline(mean_price, color='red', linestyle='--', label=f'Media £{mean_price:.2f}')
axes[0].axvline(median_price, color='green', linestyle='--', label=f'Mediana £{median_price:.2f}')
axes[0].legend()

axes[1].boxplot(df_filtered['price'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot de precios (hasta percentil 95)', fontweight='bold')
axes[1].set_ylabel('Precio (£)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{int(x):,}'))

plt.tight_layout()
plt.show()

print(f"\nNota: Se han filtrado precios superiores a £{p95:.2f} (percentil 95).")
print(f"Esto excluye {len(df) - len(df_filtered):,} productos ({(len(df)-len(df_filtered))/len(df)*100:.1f}% del total).")

In [ ]:
df_rated = df[df['stars'] > 0].copy()

mean_rating = df_rated['stars'].mean()
median_rating = df_rated['stars'].median()
mode_rating = df_rated['stars'].mode()[0]

variance_rating = df_rated['stars'].var()
std_rating = df_rated['stars'].std()
q1_rating = df_rated['stars'].quantile(0.25)
q3_rating = df_rated['stars'].quantile(0.75)
iqr_rating = q3_rating - q1_rating

skewness = df_rated['stars'].skew()
kurtosis = df_rated['stars'].kurtosis()

print(f"Análisis sobre {len(df_rated):,} productos con valoración\n")
print("MEDIDAS DE CENTRALIDAD — VALORACIONES")
print("=" * 40)
print(f"Media:    {mean_rating:.2f}")
print(f"Mediana:  {median_rating:.2f}")
print(f"Moda:     {mode_rating:.2f}")
print("\nMEDIDAS DE DISPERSIÓN — VALORACIONES")
print("=" * 40)
print(f"Varianza:     {variance_rating:.4f}")
print(f"Desv. típica: {std_rating:.4f}")
print(f"Q1:           {q1_rating:.2f}")
print(f"Q3:           {q3_rating:.2f}")
print(f"IQR:          {iqr_rating:.2f}")
print("\nFORMA DE LA DISTRIBUCIÓN")
print("=" * 40)
print(f"Asimetría (Skewness):  {skewness:.4f}")
print(f"Curtosis (Kurtosis):   {kurtosis:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df_rated['stars'], bins=20, color='steelblue', edgecolor='white')
ax.set_title('Distribución de valoraciones de productos en Amazon UK', fontweight='bold')
ax.set_xlabel('Valoración (estrellas)')
ax.set_ylabel('Número de productos')
ax.axvline(mean_rating, color='red', linestyle='--', label=f'Media {mean_rating:.2f}')
ax.axvline(median_rating, color='green', linestyle='--', label=f'Mediana {median_rating:.2f}')
ax.axvline(mode_rating, color='orange', linestyle='--', label=f'Moda {mode_rating:.2f}')
ax.legend()
plt.tight_layout()
plt.show()

## Parte 1: Análisis de Negocio – Categorías de Productos

**Top 5 categorías más listadas:**
1. Sports & Outdoors – 836,260 productos (34.2%)
2. Handmade Artwork – 19,312 productos (0.79%)
3. Handmade Jewellery – 19,229 productos (0.79%)
4. Luggage and travel gear – 19,092 productos (0.78%)
5. Handmade Home Décor – 18,978 productos (0.78%)

**Hallazgos clave:**
- Sports & Outdoors domina el catálogo con un 34.2% de todos los listings — más de 40 veces la siguiente categoría. Esta concentración extrema sugiere un sesgo en la recogida de datos hacia esta categoría o una dominancia real del mercado.
- Las categorías restantes están distribuidas de forma muy uniforme, todas en torno al 0.77–0.79%, sin una segunda categoría dominante.
- El gráfico de tarta confirma esto: Sports & Outdoors y "Otros" representan juntos el 93% de todos los productos, dejando al top 10 de categorías con solo el 7%.

**Implicación de negocio:** Cualquier vendedor o marca que quiera entrar en Amazon UK debe decidir si competir en el espacio altamente saturado de Sports & Outdoors o dirigirse a las categorías restantes, más fragmentadas pero con menor competencia.

## Parte 2: Análisis de Negocio – Precios de Productos

**Medidas de Centralidad:**
- Media: £89.24 | Mediana: £19.09 | Moda: £9.99
- La gran diferencia entre media y mediana indica una distribución con fuerte asimetría positiva, causada por productos de precio muy elevado que tiran de la media hacia arriba. La mediana (£19.09) es una medida mucho más representativa del precio "típico" de un producto en Amazon UK.
- La moda (£9.99) revela una estrategia de precios psicológicos muy extendida — los vendedores fijan precios justo por debajo de los números redondos para parecer más baratos.

**Medidas de Dispersión:**
- Una desviación típica de £345.61 sobre una mediana de £19.09 confirma una dispersión de precios extrema.
- El IQR de £36.00 (Q1: £9.99 – Q3: £45.99) muestra que el 50% de los productos se vende entre £10 y £46, un rango de precio bastante accesible.
- El rango total va de £0.01 a £100.000, lo que indica la presencia de productos de lujo o industriales junto a artículos de uso cotidiano.

**¿Por qué el histograma era difícil de leer sin filtrar?**
La distribución original tiene una asimetría positiva muy pronunciada con valores atípicos extremos (hasta £100.000). Sin filtrar, el 95% de los datos queda comprimido en una franja ilegible. Filtrando al percentil 95 (£349.51) se revela la forma real de la distribución.

**Conclusiones clave:**
- El 50% de los productos tiene un precio inferior a £46 — Amazon UK es predominantemente un marketplace de precio bajo y medio.
- El boxplot confirma la presencia masiva de outliers incluso tras filtrar al percentil 95, con el bigote extendiéndose hasta £350.
- Los vendedores que compiten en el rango £10–£46 enfrentan la mayor competencia pero también acceden a la mayor base de clientes.

**Implicación de negocio:** Las marcas que quieran entrar en Amazon UK deberían considerar estrategias de precio por debajo de £50 para capturar el mercado principal. Los productos premium por encima de £100 tienen menos competencia en listings pero una base de compradores significativamente menor.

## Parte 3: Análisis de Negocio – Valoraciones de Productos

**Nota metodológica:** El 50.2% de los productos (1,225,641) tienen 0 estrellas — productos sin ninguna valoración todavía. El análisis se realiza sobre los 1,217,990 productos que sí tienen valoración.

**Medidas de Centralidad:**
- Media: 4.32 | Mediana: 4.40 | Moda: 4.50
- Los tres valores están muy próximos y concentrados en la parte alta de la escala, lo que indica que los clientes tienden a valorar positivamente los productos que compran en Amazon UK.
- La moda de 4.5 es la valoración más frecuente, seguida de cerca por 4.6 y 4.4.

**Medidas de Dispersión:**
- La desviación típica de 0.55 sobre una escala de 1 a 5 indica valoraciones muy consistentes — poca variabilidad entre productos.
- El IQR de 0.50 (Q1: 4.10 – Q3: 4.60) confirma que el 50% central de los productos se concentra en un rango muy estrecho de media estrella.

**Forma de la distribución:**
- Asimetría de -2.38: distribución fuertemente asimétrica hacia la izquierda — la mayoría de productos se concentra en valoraciones altas, con una cola larga hacia valoraciones bajas.
- Curtosis de 9.78: distribución muy apuntada (leptocúrtica), con un pico muy pronunciado alrededor de 4.4-4.5 estrellas y colas más pesadas de lo normal.
- Las valoraciones NO siguen una distribución normal — están claramente sesgadas hacia puntuaciones altas.

**Conclusiones clave:**
- Amazon UK tiene un sesgo positivo muy marcado en las valoraciones: el 75% de los productos valorados tienen más de 4.1 estrellas.
- Las valoraciones bajas (por debajo de 3.5) son minoritarias pero representan una señal clara de problemas de producto.
- El histograma muestra un vacío notable entre 1.5 y 2.5 estrellas — los productos o son buenos (4+) o son malos (1-1.5), con poco término medio.

**Implicación de negocio:** En un marketplace donde la media es 4.32 estrellas, tener menos de 4.0 supone una desventaja competitiva significativa. Las marcas deben priorizar la gestión de reseñas y la calidad del producto para mantenerse por encima de ese umbral.